# Lightweight HBCC Full Training Pipeline

This notebook orchestrates the full workflow implemented in this repository:

1. Environment and smoke checks
2. Teacher/reference training
3. Student CE-only training
4. Short proxy architecture search
5. Technique ablations
6. Knowledge distillation
7. Structured pruning export and fine-tune
8. Benchmark matrix and Pareto summary

The default flags are conservative. Turn on the phases you want to run before executing all cells.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)

## Phase Switches

Set the flags below. For a first run, keep only `RUN_SMOKE = True`. For the full research run, enable each phase deliberately.

In [ ]:
RUN_ENV_CHECKS = True
RUN_SMOKE = True

RUN_TEACHER = False
RUN_LIGHTWEIGHT_BASELINES = False
RUN_STUDENT_CE = False
RUN_PROXY_SEARCH = False
RUN_ABLATIONS = False
RUN_KD = False
RUN_PRUNING = False
RUN_BENCHMARKS = False
RUN_PARETO_REPORT = False

FULL_EPOCHS = 200
ACCURACY_EPOCHS = 300
PROXY_EPOCHS = 30

COMMON_TRAIN_OVERRIDES = [
    # Reduce these if you hit OOM.
    # "data.batch_size=64",
    # "data.val_batch_size=128",
]

BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

# Use shorter benchmark settings while debugging the notebook.
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP = 2
DEBUG_BENCHMARK_RUNS = 3

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    """Run a repo Python command; tqdm updates are kept to one notebook line."""
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(config: str, output: str, overrides: list[str] | None = None, extra: list[str] | None = None) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args([*(overrides or []), *COMMON_TRAIN_OVERRIDES])
    args += extra or []
    run_py(args)


def benchmark(config: str, checkpoint: str | None = None, debug: bool = False, profile: bool = True) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config",
        config,
        "--output",
        "results/benchmark",
        "--batch-sizes",
        *[str(v) for v in batches],
        "--warmup",
        str(warmup),
        "--runs",
        str(runs),
    ]
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## 0. Environment and Smoke Checks

In [ ]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_latency_tiny.yaml", "--batch-size", "1"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_current_reference.yaml", "--batch-size", "1"])

if RUN_SMOKE:
    train(
        "configs/smoke.yaml",
        "runs_smoke",
        extra=["--limit-train-batches", "1", "--limit-val-batches", "1"],
    )
    benchmark("configs/smoke.yaml", debug=True, profile=False)

## 1. Teacher and Reference Baselines

Train ResNet-18 first. It is both the accuracy reference and the default KD teacher.

In [ ]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        "runs_teacher",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

## 2. Student CE-only Training

This is the fair baseline for each HBCC student before applying KD or pruning.

In [ ]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_students",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/hbcc_latency_small.yaml",
        "runs_students",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )

## 3. Short Proxy Architecture Search

Use 20-50 epochs to reject weak configurations. Only top candidates should get full training.

In [ ]:
PROXY_JOBS = [
    {
        "name": "proxy_dims24_depth1111_mlp2",
        "overrides": [
            "model.embed_dims=[24,48,96,160]",
            "model.depths=[1,1,1,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_depth1121_mlp2",
        "overrides": [
            "model.embed_dims=[32,48,96,160]",
            "model.depths=[1,1,2,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_64_depth1221_mlp3",
        "overrides": [
            "model.embed_dims=[32,64,128,192]",
            "model.depths=[1,2,2,1]",
            "model.mlp_ratios=3.0",
            "model.heads=[1,2,4,4]",
        ],
    },
]

if RUN_PROXY_SEARCH:
    for job in PROXY_JOBS:
        train(
            "configs/hbcc_latency_tiny.yaml",
            "runs_proxy",
            overrides=[
                f"experiment.name={job['name']}",
                f"train.epochs={PROXY_EPOCHS}",
                *job["overrides"],
            ],
        )

## 4. Independent Ablations

Run each technique independently and decide `keep`, `drop`, or `defer` from metrics.

In [ ]:
ABLATION_CONFIGS = [
    "configs/ablations/hbcc_tiny_hamming_late.yaml",
    "configs/ablations/hbcc_tiny_lbpconv.yaml",
]

if RUN_ABLATIONS:
    for cfg in ABLATION_CONFIGS:
        train(cfg, "runs_ablations", overrides=[f"train.epochs={FULL_EPOCHS}"])

## 5. Knowledge Distillation

Requires a trained teacher checkpoint. The teacher is used only during training; inference latency of the student is unchanged.

In [ ]:
TEACHER_CHECKPOINT = "runs_teacher/resnet18_cifar/best.pth"

if RUN_KD:
    if not (ROOT / TEACHER_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train the teacher first: {TEACHER_CHECKPOINT}")
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_kd",
        overrides=[
            f"train.epochs={FULL_EPOCHS}",
            "train.kd_alpha=0.5",
            "train.kd_temperature=4.0",
        ],
        extra=[
            "--teacher-config",
            "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint",
            TEACHER_CHECKPOINT,
        ],
    )

## 6. Structured Pruning Export

Do not judge latency from the soft-mask model. Export a materialized smaller config, fine-tune it, then benchmark the exported model.

In [ ]:
PRUNING_CHECKPOINT = "runs_pruning/hbcc_tiny_pruning_mask/best.pth"
PRUNED_CONFIG = "configs/generated_ablations/hbcc_tiny_pruned_export.yaml"

if RUN_PRUNING:
    train(
        "configs/ablations/hbcc_tiny_pruning_mask.yaml",
        "runs_pruning",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    run_py([
        "tools/export_pruned.py",
        "--config",
        "configs/ablations/hbcc_tiny_pruning_mask.yaml",
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--output",
        PRUNED_CONFIG,
        "--threshold",
        "0.5",
    ])
    train(
        PRUNED_CONFIG,
        "runs_pruned_finetune",
        overrides=[f"train.epochs={max(20, FULL_EPOCHS // 10)}"],
    )

## 7. Benchmark Matrix

Run this after training checkpoints exist. Missing checkpoints are skipped and the benchmark falls back to untrained weights only when no checkpoint is provided.

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml", "runs_teacher/resnet18_cifar/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml", "runs_baselines/mobilenet_v2_cifar/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", "runs_baselines/shufflenet_v2_x1_0_cifar/best.pth"),
    ("configs/coc_cifar_baseline.yaml", None),
    ("configs/hbcc_current_reference.yaml", None),
    ("configs/hbcc_latency_tiny.yaml", "runs_students/hbcc_latency_tiny/best.pth"),
    ("configs/hbcc_latency_small.yaml", "runs_students/hbcc_latency_small/best.pth"),
    ("configs/hbcc_accuracy_small.yaml", "runs_accuracy/hbcc_accuracy_small/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml", "runs_accuracy/hbcc_accuracy_medium/best.pth"),
    ("configs/hbcc_latency_tiny.yaml", "runs_kd/hbcc_latency_tiny/best.pth"),
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "runs_ablations/hbcc_tiny_hamming_late/best.pth"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml", "runs_ablations/hbcc_tiny_lbpconv/best.pth"),
    (PRUNED_CONFIG, "runs_pruned_finetune/hbcc_tiny_pruned_export/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, checkpoint=ckpt, debug=False, profile=True)

## 8. Read Metrics and Build Summary Tables

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for metrics_path in ROOT.glob("runs*/**/metrics.jsonl"):
        records = read_jsonl(metrics_path)
        if not records:
            continue
        best = max(records, key=lambda r: r.get("val_acc1", -1))
        rows.append({
            "run_dir": str(metrics_path.parent.relative_to(ROOT)),
            "experiment": metrics_path.parent.name,
            "best_epoch": best.get("epoch"),
            "best_val_acc1": best.get("val_acc1"),
            "best_val_loss": best.get("val_loss"),
            "last_epoch": records[-1].get("epoch"),
            "last_train_acc1": records[-1].get("train_acc1"),
        })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / "results" / "benchmark").glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rec["benchmark_file"] = str(path.relative_to(ROOT))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = [
    "config_id",
    "model_name",
    "params_total",
    "macs",
    "bops",
    "latency_ms_b1",
    "throughput_b16",
    "throughput_b64",
    "throughput_b128",
    "peak_memory_mb",
]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

## 9. Pareto Summary

This cell combines training accuracy with benchmark records when names match. Review the mapping before using it as a final report.

In [ ]:
def attach_accuracy(bench: pd.DataFrame, train_metrics: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    if train_metrics.empty:
        return out
    acc_map = dict(zip(train_metrics["experiment"], train_metrics["best_val_acc1"]))
    for idx, row in out.iterrows():
        name = row.get("config_id")
        out.at[idx, "acc1"] = acc_map.get(name)
    return out


def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:
    required = ["acc1", "params_total", "latency_ms_b1", "peak_memory_mb"]
    if df.empty or any(c not in df.columns for c in required):
        return pd.DataFrame()
    valid = df.dropna(subset=required).copy()
    keep = []
    for i, row in valid.iterrows():
        dominated = False
        for j, other in valid.iterrows():
            if i == j:
                continue
            better_or_equal = (
                other["acc1"] >= row["acc1"]
                and other["params_total"] <= row["params_total"]
                and other["latency_ms_b1"] <= row["latency_ms_b1"]
                and other["peak_memory_mb"] <= row["peak_memory_mb"]
            )
            strictly_better = (
                other["acc1"] > row["acc1"]
                or other["params_total"] < row["params_total"]
                or other["latency_ms_b1"] < row["latency_ms_b1"]
                or other["peak_memory_mb"] < row["peak_memory_mb"]
            )
            if better_or_equal and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return valid.loc[keep].sort_values(["acc1", "latency_ms_b1"], ascending=[False, True])


combined_df = attach_accuracy(bench_df, train_df)
frontier_df = pareto_frontier(combined_df)

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
if not combined_df.empty:
    combined_df.to_csv(out_dir / "combined_training_benchmark.csv", index=False)
if not frontier_df.empty:
    frontier_df.to_csv(out_dir / "pareto_frontier.csv", index=False)

frontier_df[[c for c in ["config_id", "acc1", "params_total", "macs", "bops", "latency_ms_b1", "peak_memory_mb"] if c in frontier_df.columns]]

In [ ]:
if RUN_PARETO_REPORT:
    run_py(["tools/pareto_report.py", "results/benchmark", "--output", "results/pareto.md"])

## Recommended Execution Order

1. Run environment checks and smoke.
2. Enable `RUN_TEACHER` and train ResNet-18.
3. Enable `RUN_STUDENT_CE` and train HBCC Tiny/Small plus HBCC-Accuracy Small/Medium.
4. Enable `RUN_PROXY_SEARCH`; use the table to choose top candidates.
5. Enable `RUN_ABLATIONS` for Hamming/LBPConv.
6. Enable `RUN_KD` after the teacher checkpoint exists.
7. Enable `RUN_PRUNING` for mask training, export, and fine-tune.
8. Enable `RUN_BENCHMARKS` and build final tables.

Do not treat proxy accuracy or untrained benchmark records as final scientific evidence.